In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from transformers import (
	AutoTokenizer,
	AutoModelForSequenceClassification,
	get_linear_schedule_with_warmup,
)
from tqdm import tqdm
import re

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [2]:
train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')

print(f'Train: {len(train_df)} rows | Test: {len(test_df)} rows')
print(train_df['label'].value_counts().sort_index())
train_df.head(3)

Train: 252000 rows | Test: 168000 rows
label
0    50400
1    50400
2    50400
3    50400
4    50400
Name: count, dtype: int64


,id,sentence,label
0,0,Moderner Weihnachtsbaum in weiß\n\nDieses Jahr...,4
1,1,Passt wie angegossen\n\nTasche kam schnell und...,4
2,2,Schlechte Qualität\n\nIch habe sehr lange auf ...,1


In [3]:
def clean_text(text):
	text = re.sub(r'\s+', ' ', text).strip() # get rid of extra whitespace
	return text

train_df['sentence'] = train_df['sentence'].apply(clean_text)
test_df['sentence']  = test_df['sentence'].apply(clean_text)

In [4]:
train_sentences, val_sentences, train_labels, val_labels = train_test_split(
	train_df['sentence'].tolist(),
	train_df['label'].tolist(),
	test_size=0.10,
	random_state=SEED,
	stratify=train_df['label'].tolist()
)
print(f'Train split: {len(train_sentences)} | Val split: {len(val_sentences)}')

Train split: 226800 | Val split: 25200


In [5]:
MODEL_NAME = 'nlptown/bert-base-multilingual-uncased-sentiment'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [6]:
os.makedirs("data/bert-encodings", exist_ok=True)

if not os.path.exists("data/bert-encodings/bert_train_encodings.pt"):
	train_encodings = tokenizer(train_sentences, truncation=True, padding=True, max_length=512, return_tensors='pt')
	val_encodings = tokenizer(val_sentences, truncation=True, padding=True, max_length=512, return_tensors='pt')
	test_encodings = tokenizer(test_df['sentence'].tolist(), truncation=True, padding=True, max_length=512, return_tensors='pt')
	
	torch.save(train_encodings, "data/bert-encodings/bert_train_encodings.pt")
	torch.save(val_encodings, "data/bert-encodings/bert_val_encodings.pt")
	torch.save(test_encodings, "data/bert-encodings/bert_test_encodings.pt")
else:
	train_encodings = torch.load("data/bert-encodings/bert_train_encodings.pt", weights_only=False)
	val_encodings = torch.load("data/bert-encodings/bert_val_encodings.pt", weights_only=False)
	test_encodings = torch.load("data/bert-encodings/bert_test_encodings.pt", weights_only=False)

In [7]:
class ReviewsDataset(Dataset):
	def __init__(self, encodings, labels):
		self.encodings = encodings
		self.labels = labels

	def __getitem__(self, idx):
		#item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()} # input_ids, attention_mask are the most important
		item = {}
		item['input_ids'] =  torch.tensor(self.encodings["input_ids"][idx])
		item['attention_mask'] = torch.tensor(self.encodings["attention_mask"][idx])
		item['labels'] = torch.tensor(self.labels[idx])
		return item

	def __len__(self):
		return len(self.labels)

train_dataset = ReviewsDataset(train_encodings, train_labels)
val_dataset = ReviewsDataset(val_encodings, val_labels)
test_dataset = ReviewsDataset(test_encodings, [-1] * len(test_df)) # dummy labels for test set

random_indices_train = random.sample(range(len(train_dataset)), 4000) # 226k
random_indices_val = random.sample(range(len(val_dataset)), 500) # 25k

train_subset = Subset(train_dataset, random_indices_train)
val_subset = Subset(val_dataset, random_indices_val)
# test_subset = Subset(test_dataset, range(500))

train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)
model.to(device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(105879, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [9]:
def evaluation_score(preds, labels):
	mae = np.mean(np.abs(np.array(preds) - np.array(labels)))
	return 1.0 - mae / 4.0

def train_epoch(model, dataloader, optimizer, scheduler=None):
	model.train()
	epoch_loss = 0.0
	num_batches = len(dataloader)
	predictions = []
	labels = []

	for batch in dataloader:
		batch = {k: v.to(device) for k, v in batch.items()}

		outputs = model(**batch)

		predicted_classes = torch.argmax(outputs.logits, dim=1)
		predicted_classes = predicted_classes.cpu().detach().numpy()

		predictions.extend(predicted_classes)
		labels.extend(batch['labels'].cpu().detach().numpy())

		loss = outputs.loss # we use the model's loss
		loss_scalar = loss.item()
		loss.backward()

		optimizer.step()
		optimizer.zero_grad()

		if scheduler:
			scheduler.step()
		epoch_loss += loss_scalar

	epoch_loss /= num_batches
	return epoch_loss, predictions, labels

def eval_epoch(model, dataloader):
	model.eval()
	epoch_loss = 0.0
	num_batches = len(dataloader)
	predictions = []
	labels = []

	with torch.no_grad():
		for batch in dataloader:
			batch = {k: v.to(device) for k, v in batch.items()}

			outputs = model(**batch)

			predicted_classes = torch.argmax(outputs.logits, dim=1)
			predicted_classes = predicted_classes.cpu().detach().numpy()

			predictions.extend(predicted_classes)
			labels.extend(batch['labels'].cpu().detach().numpy())

			loss = outputs.loss # we use the model's loss
			loss_scalar = loss.item()
			epoch_loss += loss_scalar

	epoch_loss /= num_batches
	return epoch_loss, predictions, labels

In [ ]:
hyperparams = {
	'num_epochs': 5,
	'lr': 1e-5,
	'step_size': None,
	'gamma': None,
}
hyperparams_str = "%".join(f"{k}-{v}" for k, v in hyperparams.items())
os.makedirs("checkpoints", exist_ok=True)


optimizer = AdamW(model.parameters(), lr=hyperparams['lr'])
#scheduler = StepLR(optimizer, step_size=hyperparams['step_size'], gamma=hyperparams['gamma'])
scheduler = None

train_losses, val_losses = [], []
train_scores, val_scores = [], []
best_val_acc = 0.0

for epoch_idx in tqdm(range(hyperparams['num_epochs'])):
	train_epoch_loss, train_predictions, train_labels = train_epoch(model, train_loader, optimizer, scheduler)
	val_epoch_loss, val_predictions, val_labels = eval_epoch(model, val_loader)

	train_losses.append(train_epoch_loss)
	val_losses.append(val_epoch_loss)

	train_epoch_score = evaluation_score(train_predictions, train_labels)
	val_epoch_score = evaluation_score(val_predictions, val_labels)
	train_scores.append(train_epoch_score)
	val_scores.append(val_epoch_score)

	if val_epoch_score > best_val_acc:
		best_val_acc = val_epoch_score
		torch.save(model.state_dict(), f'./checkpoints/best_model_init_{hyperparams_str}.pth')

	print("epoch %d, train loss=%f, train acc=%f, val loss=%f, val acc=%f" % (
		epoch_idx,
		train_epoch_loss,
		train_epoch_score,
		val_epoch_loss,
		val_epoch_score
	))

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
best_model_path = 'checkpoints/best_model_init_num_epochs-5%lr-1e-05%step_size-None%gamma-None_1_epoch.pth'
model.load_state_dict(torch.load(best_model_path))
model.to(device)
model.eval()

test_predictions = []
with torch.no_grad():
	for batch in tqdm(test_loader):
		batch = {k: v.to(device) for k, v in batch.items()}

		outputs = model(**batch)

		predicted_classes = torch.argmax(outputs.logits, dim=1)
		predicted_classes = predicted_classes.cpu().detach().numpy()

		test_predictions.extend(predicted_classes)


submission_df = pd.DataFrame({
	'id': test_df['id'],
	'label': test_predictions
})
assert len(submission_df) == len(test_df)
assert submission_df['label'].isin([0,1,2,3,4]).all()

submission_output_path = f'data/submission_{best_model_path}.csv'
submission_df.to_csv(submission_output_path, index=False)
print(f'Submission saved to {submission_output_path}')
print(submission_df.head(3))

  0%|          | 0/10500 [00:00<?, ?it/s]